Cell 1: Initial Setup and Imports

In [1]:
# Cell 1: Initial Setup and Imports
# -*- coding: utf-8 -*-
"""
INSTANCE_Temporal_Event_Based_Split_Experiment.ipynb

This notebook runs a single temporal event-based split experiment on the INSTANCE dataset.
The key difference from random splits: events are split chronologically based on their
occurrence time, maintaining temporal order.
"""

# Import required libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import json
import os
import time
import random
import seaborn as sns
from tqdm import tqdm
from google.colab import drive
import pickle

# Helper function to convert numpy types to Python types for JSON serialisation
def numpy_to_python(obj):
    """Convert numpy types to Python types for JSON serialisation."""
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, dict):
        return {k: numpy_to_python(v) for k, v in obj.items()}
    elif isinstance(obj, list) or isinstance(obj, tuple):
        return [numpy_to_python(i) for i in obj]
    else:
        return obj

# Mount Google Drive if using Colab
drive.mount('/content/drive')

# Configure environment
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Record start time
start_time = time.time()

# Define paths to data files
base_dir = "/content/drive/My Drive/2023-2024/UCL MSc in DSML/Term 3/MSc Project/Code/INSTANCE_Event_Based"
all_data_file = os.path.join(base_dir, "all_data.pt")
all_labels_file = os.path.join(base_dir, "all_labels.pt")
split_info_file = os.path.join(base_dir, "event_split_info.pkl")
output_dir = os.path.join(base_dir, "temporal_experiment_results")
os.makedirs(output_dir, exist_ok=True)

# Check if files exist
assert os.path.isfile(all_data_file), f"Data file not found at {all_data_file}"
assert os.path.isfile(all_labels_file), f"Labels file not found at {all_labels_file}"
assert os.path.isfile(split_info_file), f"Split info file not found at {split_info_file}"

print("✓ INSTANCE event-based data files found")
print(f"✓ Output directory: {output_dir}")
print("\n" + "="*80)
print("TEMPORAL EVENT-BASED SPLITTING EXPERIMENT")
print("Events split chronologically - no random shuffling")
print("="*80)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
✓ INSTANCE event-based data files found
✓ Output directory: /content/drive/My Drive/2023-2024/UCL MSc in DSML/Term 3/MSc Project/Code/INSTANCE_Event_Based/temporal_experiment_results

TEMPORAL EVENT-BASED SPLITTING EXPERIMENT
Events split chronologically - no random shuffling


Cell 2: Dataset and Model Classes

In [2]:
# Cell 2: Dataset and Model Classes
class EarthquakeDataset(Dataset):
    """Dataset class for earthquake data."""
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

class EarthquakeModel(nn.Module):
    """MagNet architecture for earthquake magnitude estimation - ADAPTED FOR INSTANCE FORMAT."""
    def __init__(self):
        super(EarthquakeModel, self).__init__()
        self.conv1 = nn.Conv1d(3, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(64, 32, kernel_size=3, padding=1)
        self.maxpool = nn.MaxPool1d(4, padding=1)
        self.dropout = nn.Dropout(0.2)
        self.lstm = nn.LSTM(32, 100, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(200, 2)  # Output: [magnitude_prediction, log_variance]

    def forward(self, x):
        # INSTANCE data format: [batch, channels, time_steps] - NO TRANSPOSE NEEDED
        # First conv block
        x = self.conv1(x)
        x = self.dropout(x)
        x = self.maxpool(x)

        # Second conv block
        x = self.conv2(x)
        x = self.dropout(x)
        x = self.maxpool(x)

        # Prepare for LSTM: [batch, time_steps, features]
        x = x.transpose(1, 2)

        # LSTM layer
        x, _ = self.lstm(x)

        # Get the last output of the LSTM
        x = x[:, -1, :]

        # Output layer with magnitude prediction and uncertainty
        x = self.fc(x)

        return x

Cell 3: Training Components

In [3]:
# Cell 3: Training Components
class EarlyStopping:
    """Early stopping to prevent overfitting."""
    def __init__(self, patience=7, verbose=False, delta=0, run_id=None,
                 split_num=None, model_seed=None):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = float('inf')
        self.delta = delta
        self.run_id = run_id
        self.split_num = split_num
        self.model_seed = model_seed
        self.best_model_path = None

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            print(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f})')
        self.best_model_path = os.path.join(
            output_dir, f'best_model_temporal_seed_{self.model_seed}.pth'
        )
        torch.save(model.state_dict(), self.best_model_path)
        self.val_loss_min = val_loss

def custom_loss(y_pred, y_true):
    """
    Custom loss function combining prediction error and uncertainty.
    L = 0.5 * exp(-s) * (y_true - y_hat)^2 + 0.5 * s
    """
    y_hat = y_pred[:, 0]    # Predicted magnitude
    s = y_pred[:, 1]        # Predicted log variance (uncertainty)

    # Compute loss
    loss = 0.5 * torch.exp(-s) * (y_true - y_hat)**2 + 0.5 * s

    return torch.mean(loss)

Cell 4: Training and Evaluation Functions

In [4]:
# Cell 4: Training and Evaluation Functions
def set_seed(seed):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def train_model(model, train_loader, val_loader, num_epochs=300, patience=5,
                model_seed=None, verbose=False):
    """Train the model with early stopping and learning rate scheduling."""
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=np.sqrt(0.1),
        cooldown=0, patience=4, min_lr=0.5e-6
    )

    early_stopping = EarlyStopping(
        patience=patience, verbose=verbose,
        model_seed=model_seed
    )

    criterion = custom_loss
    train_losses = []
    val_losses = []

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            outputs = model(data)
            loss = criterion(outputs, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            running_loss += loss.item()

        # Validation phase
        val_loss = 0.0
        model.eval()
        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device)
                outputs = model(data)
                loss = criterion(outputs, target)
                val_loss += loss.item()

        # Calculate average losses
        val_loss /= len(val_loader)
        running_loss /= len(train_loader)

        # Learning rate scheduling and early stopping
        scheduler.step(val_loss)
        early_stopping(val_loss, model)

        if verbose:
            print(f'Epoch {epoch+1}, Loss: {running_loss:.4f}, '
                  f'Validation Loss: {val_loss:.4f}, '
                  f'LR: {optimizer.param_groups[0]["lr"]:.6f}')

        train_losses.append(running_loss)
        val_losses.append(val_loss)

        if early_stopping.early_stop:
            if verbose:
                print(f'Early stopping triggered at epoch {epoch+1}')
            break

    return {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'best_model_path': early_stopping.best_model_path
    }

def estimate_uncertainty(model, data_loader, num_samples=50):
    """Estimate model uncertainty using Monte Carlo dropout."""
    model.eval()

    # Enable dropout during inference for Monte Carlo sampling
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()

    predictions = []
    log_variances = []

    with torch.no_grad():
        for _ in range(num_samples):
            batch_predictions = []
            batch_log_variances = []
            for data, _ in data_loader:
                data = data.to(device)
                output = model(data)
                batch_predictions.append(output[:, 0].cpu().numpy())
                batch_log_variances.append(output[:, 1].cpu().numpy())
            predictions.append(np.concatenate(batch_predictions))
            log_variances.append(np.concatenate(batch_log_variances))

    predictions = np.array(predictions)
    log_variances = np.array(log_variances)

    # Calculate uncertainties
    mean_prediction = np.mean(predictions, axis=0)
    yhat_squared_mean = np.mean(np.square(predictions), axis=0)
    aleatoric_uncertainty = np.mean(np.exp(log_variances), axis=0)
    epistemic_uncertainty = np.std(predictions, axis=0)
    combined_uncertainty = yhat_squared_mean - np.square(mean_prediction) + aleatoric_uncertainty

    return mean_prediction, epistemic_uncertainty, aleatoric_uncertainty, combined_uncertainty

def evaluate_model(model_path, test_loader):
    """Evaluate a trained model on test data."""
    model = EarthquakeModel().to(device)
    model.load_state_dict(torch.load(model_path))

    # Get predictions and uncertainties
    mean_pred, epistemic_unc, aleatoric_unc, combined_unc = estimate_uncertainty(model, test_loader)

    # Get true values
    true_values = []
    for _, target in test_loader:
        true_values.append(target.numpy())
    true_values = np.concatenate(true_values)

    # Calculate MAE
    mae = np.mean(np.abs(mean_pred - true_values))

    return {
        'mae': float(mae),
        'mean_prediction': mean_pred,
        'true_values': true_values,
        'epistemic_uncertainty': epistemic_unc,
        'aleatoric_uncertainty': aleatoric_unc,
        'combined_uncertainty': combined_unc,
        'mean_epistemic_uncertainty': float(np.mean(epistemic_unc)),
        'mean_aleatoric_uncertainty': float(np.mean(aleatoric_unc)),
        'mean_combined_uncertainty': float(np.mean(combined_unc))
    }

Cell 5: Temporal Split Function (KEY MODIFICATION)

In [5]:
# Cell 5: Temporal Split Function (KEY MODIFICATION)
def create_temporal_event_based_split():
    """
    Create a TEMPORAL event-based split.

    KEY DIFFERENCE FROM RANDOM SPLITS:
    - Events are split based on chronological order
    - No shuffling of events
    - Maintains temporal ordering: train (earliest) -> val -> test (latest)

    Returns:
        Dictionary with train, val, test data and labels
    """
    # Load the data
    all_data = torch.load(all_data_file)
    all_labels = torch.load(all_labels_file)

    # Load the split information
    with open(split_info_file, 'rb') as f:
        split_info = pickle.load(f)

    unique_events = split_info['unique_events']
    event_indices = split_info['event_indices']
    train_ratio = split_info['train_ratio']  # 0.7
    val_ratio = split_info['val_ratio']      # 0.1

    # TEMPORAL SPLIT: Use events in their original chronological order
    # NO SHUFFLING - events are already sorted by time from preprocessing
    print("\nCreating TEMPORAL split (chronological order):")
    print(f"   Total unique events: {len(unique_events)}")

    # Calculate split sizes
    train_size = int(train_ratio * len(unique_events))
    val_size = int(val_ratio * len(unique_events))

    # Split events chronologically
    train_events = unique_events[:train_size]  # Earliest 70% of events
    val_events = unique_events[train_size:train_size + val_size]  # Next 10%
    test_events = unique_events[train_size + val_size:]  # Latest 20%

    print(f"   Train: First {len(train_events)} events (earliest {train_ratio*100:.0f}%)")
    print(f"   Val: Next {len(val_events)} events (middle {val_ratio*100:.0f}%)")
    print(f"   Test: Last {len(test_events)} events (latest {(1-train_ratio-val_ratio)*100:.0f}%)")

    # Collect indices for each split
    train_indices = np.concatenate([event_indices[event_id] for event_id in train_events])
    val_indices = np.concatenate([event_indices[event_id] for event_id in val_events])
    test_indices = np.concatenate([event_indices[event_id] for event_id in test_events])

    # Extract data using the indices
    train_data = all_data[train_indices]
    train_labels = all_labels[train_indices]

    val_data = all_data[val_indices]
    val_labels = all_labels[val_indices]

    test_data = all_data[test_indices]
    test_labels = all_labels[test_indices]

    return {
        'train_data': train_data,
        'train_labels': train_labels,
        'val_data': val_data,
        'val_labels': val_labels,
        'test_data': test_data,
        'test_labels': test_labels,
        'train_events': train_events,
        'val_events': val_events,
        'test_events': test_events,
        'split_type': 'temporal'
    }

Cell 6: Run Temporal Experiment

In [6]:
# Cell 6: Run Temporal Experiment
def run_temporal_experiment(model_seeds):
    """
    Run the temporal split experiment with multiple model initialisations.

    Args:
        model_seeds: List of random seeds for model initialisation

    Returns:
        Dictionary with experiment results
    """
    print("\nRunning TEMPORAL event-based splitting experiment")
    print("="*50)

    # Create the temporal data split
    split_data = create_temporal_event_based_split()

    # Create datasets
    train_dataset = EarthquakeDataset(split_data['train_data'], split_data['train_labels'])
    val_dataset = EarthquakeDataset(split_data['val_data'], split_data['val_labels'])
    test_dataset = EarthquakeDataset(split_data['test_data'], split_data['test_labels'])

    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False, num_workers=2)

    # Log split sizes
    print(f"\nDataset Statistics:")
    print(f"   Train: {len(train_dataset)} samples from {len(split_data['train_events'])} events")
    print(f"   Validation: {len(val_dataset)} samples from {len(split_data['val_events'])} events")
    print(f"   Test: {len(test_dataset)} samples from {len(split_data['test_events'])} events")

    # Run experiments with multiple random initialisations
    seed_results = []

    print(f"\nTraining with {len(model_seeds)} different model initialisations...")
    print("-"*50)

    for model_seed in model_seeds:
        print(f"\nModel seed {model_seed}:")

        # Set random seed for model initialisation
        set_seed(model_seed)

        # Initialise the model
        model = EarthquakeModel().to(device)

        # Train the model
        training_result = train_model(
            model, train_loader, val_loader,
            model_seed=model_seed,
            verbose=False  # Set to True for detailed progress
        )

        # Evaluate the model
        best_model_path = training_result['best_model_path']
        evaluation_result = evaluate_model(best_model_path, test_loader)

        # Store results
        seed_results.append({
            'model_seed': model_seed,
            'training_history': {
                'train_losses': training_result['train_losses'],
                'val_losses': training_result['val_losses']
            },
            'evaluation': evaluation_result
        })

        print(f"   ✓ MAE: {evaluation_result['mae']:.4f}")
        print(f"   ✓ Aleatoric Uncertainty: {evaluation_result['mean_aleatoric_uncertainty']:.4f}")
        print(f"   ✓ Epistemic Uncertainty: {evaluation_result['mean_epistemic_uncertainty']:.4f}")

    # Find median performance
    sorted_results = sorted(seed_results, key=lambda x: x['evaluation']['mae'])
    median_result = sorted_results[len(model_seeds) // 2]

    return {
        'split_type': 'temporal',
        'all_seed_results': seed_results,
        'median_mae': median_result['evaluation']['mae'],
        'median_model_seed': median_result['model_seed'],
        'median_aleatoric_uncertainty': median_result['evaluation']['mean_aleatoric_uncertainty'],
        'median_epistemic_uncertainty': median_result['evaluation']['mean_epistemic_uncertainty'],
        'median_combined_uncertainty': median_result['evaluation']['mean_combined_uncertainty'],
        'train_size': len(train_dataset),
        'val_size': len(val_dataset),
        'test_size': len(test_dataset),
        'train_events': len(split_data['train_events']),
        'val_events': len(split_data['val_events']),
        'test_events': len(split_data['test_events'])
    }

Cell 7: Main Execution

In [ ]:
# Cell 7: Main Execution
if __name__ == "__main__":
    # Define model initialisation seeds (same as in random splits for fair comparison)
    model_seeds = [42, 123, 256, 789, 1024]  # 5 different model initialisations

    # Define results file for temporal experiment
    results_file = os.path.join(output_dir, "temporal_split_results.json")

    print("\n" + "="*80)
    print("INSTANCE TEMPORAL EVENT-BASED SPLITTING EXPERIMENT")
    print("="*80)
    print("\nExperiment Configuration:")
    print(f"   • Split Type: Temporal (chronological)")
    print(f"   • Split Ratio: 70% train, 10% val, 20% test")
    print(f"   • Model Seeds: {model_seeds}")
    print(f"   • Device: {device}")

    # Run the temporal split experiment
    result = run_temporal_experiment(model_seeds)

    # Save results
    with open(results_file, 'w') as f:
        serializable_results = numpy_to_python(result)
        json.dump(serializable_results, f, indent=4)

    # End timing
    end_time = time.time()
    elapsed_time = end_time - start_time

    # Print summary
    print("\n" + "="*80)
    print("EXPERIMENT COMPLETED SUCCESSFULLY")
    print("="*80)
    print(f"\nTEMPORAL SPLIT RESULTS:")
    print(f"   • Median MAE: {result['median_mae']:.4f}")
    print(f"   • Median Aleatoric Uncertainty: {result['median_aleatoric_uncertainty']:.4f}")
    print(f"   • Median Epistemic Uncertainty: {result['median_epistemic_uncertainty']:.4f}")
    print(f"   • Median Combined Uncertainty: {result['median_combined_uncertainty']:.4f}")
    print(f"   • Best performing seed: {result['median_model_seed']}")

    print(f"\nTotal execution time: {elapsed_time/60:.2f} minutes")
    print(f"\nResults saved to: {results_file}")

    print("\n" + "="*80)
    print("ANALYSIS INSIGHTS:")
    print("="*80)
    print("Temporal splitting tests the model's ability to generalise to future events.")
    print("Compare this MAE with random event-based splits to assess:")
    print("  1. Temporal distribution shift impact")
    print("  2. Model's robustness to evolving seismic patterns")
    print("  3. Practical deployment scenario performance")
    print("\nExpected: Temporal MAE may be higher than random splits due to:")
    print("  - Distribution shift over time")
    print("  - Potential changes in earthquake characteristics")
    print("  - Station/instrument changes over time")


INSTANCE TEMPORAL EVENT-BASED SPLITTING EXPERIMENT

Experiment Configuration:
   • Split Type: Temporal (chronological)
   • Split Ratio: 70% train, 10% val, 20% test
   • Model Seeds: [42, 123, 256, 789, 1024]
   • Device: cuda

Running TEMPORAL event-based splitting experiment

Creating TEMPORAL split (chronological order):
   Total unique events: 34031
   Train: First 23821 events (earliest 70%)
   Val: Next 3403 events (middle 10%)
   Test: Last 6807 events (latest 20%)

Dataset Statistics:
   Train: 218404 samples from 23821 events
   Validation: 66008 samples from 3403 events
   Test: 77822 samples from 6807 events

Training with 5 different model initialisations...
--------------------------------------------------

Model seed 42:
   ✓ MAE: 0.2387
   ✓ Aleatoric Uncertainty: 0.1206
   ✓ Epistemic Uncertainty: 0.0867

Model seed 123:
   ✓ MAE: 0.2066
   ✓ Aleatoric Uncertainty: 0.0815
   ✓ Epistemic Uncertainty: 0.0670

Model seed 256:
   ✓ MAE: 0.2337
   ✓ Aleatoric Uncertainty